In [1]:
import os, yaml, sys
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output
import random
from scipy.io import loadmat
from scipy.stats import spearmanrho
ENV = os.getenv("MY_ENV", "dev")
with open("../../config.yaml", "r") as f:
    config = yaml.safe_load(f)
paths = config[ENV]["paths"]
sys.path.append(paths["src_path"])
sys.path.append(paths["useful_stuff_path"])
from useful_stuff.general_utils.RSA import RSA
from project_specific_utils.dataloader import load_meg_data, load_concat_regressout_meg
from analyses.subsampling_lagged_comparisons import get_spaced_pseudotrials


In [42]:
from dataclasses import dataclass, field

@dataclass
class Cfg:
    sub_num =3
    neu_fs = 100
    gaze_fs = 50
    mod_fs = 23.976
    repetition = 0
    sensors_group = 'occ'
    regress_out_gaze = "PCR"
    PCs_to_regress_out = 50
    timepts_to_regress_out = 50
    pseudotrials_len_tps = 1
    pseudotrials_n = 20
    min_separation = 5
    neural_metric = "correlation"
    iterations_n = 1000

cfg = Cfg()
model_len = [round(i*cfg.neu_fs/config["movie_fs"]) for i in config["model_len"]]
subjects = config["subjects"][:10]

In [32]:
i_sub = 3; repetition = 0; rank=0; 

In [33]:
n = load_concat_regressout_meg(paths, i_sub, repetition, cfg.sensors_group, cfg.neu_fs, cfg.gaze_fs, cfg.regress_out_gaze, cfg.PCs_to_regress_out, timepts_to_regress_out=(-cfg.timepts_to_regress_out,cfg.timepts_to_regress_out) , rank=rank)
pseudotrials_idx = get_spaced_pseudotrials(n, cfg.pseudotrials_len_tps, cfg.pseudotrials_n, 5, model_len,) # TODO decide how to get the signal here (or its length)

15:26:02 - rank 0 Loading MEG signal: regress_out_gaze='PCR'
15:26:03 - rank 0 Shape runs [1 2 3]: [(41, 87379), (41, 86278), (41, 79071)]


In [19]:
subjects_signal = []
for i_sub in subjects:
    n = load_concat_regressout_meg(paths, i_sub, repetition, cfg.sensors_group, cfg.neu_fs, cfg.gaze_fs, cfg.regress_out_gaze, cfg.PCs_to_regress_out, timepts_to_regress_out=(-cfg.timepts_to_regress_out,cfg.timepts_to_regress_out) , rank=rank)
    subjects_signal.append(n)

15:18:42 - rank 0 Loading MEG signal: regress_out_gaze='PCR'
15:18:43 - rank 0 Shape runs [1 2 3]: [(41, 87379), (41, 86278), (41, 79071)]
15:18:43 - rank 0 Loading MEG signal: regress_out_gaze='PCR'
15:18:45 - rank 0 Shape runs [1 2 3]: [(41, 87379), (41, 86278), (41, 79071)]
15:18:45 - rank 0 Loading MEG signal: regress_out_gaze='PCR'
15:18:46 - rank 0 Shape runs [1 2 3]: [(41, 87379), (41, 86278), (41, 79071)]
15:18:46 - rank 0 Loading MEG signal: regress_out_gaze='PCR'
15:18:47 - rank 0 Shape runs [1 2 3]: [(41, 87379), (41, 86278), (41, 79071)]
15:18:47 - rank 0 Loading MEG signal: regress_out_gaze='PCR'
15:18:48 - rank 0 Shape runs [1 2 3]: [(41, 87379), (41, 86278), (41, 79071)]
15:18:48 - rank 0 Loading MEG signal: regress_out_gaze='PCR'
15:18:49 - rank 0 Shape runs [1 2 3]: [(41, 87379), (41, 86278), (41, 79071)]
15:18:49 - rank 0 Loading MEG signal: regress_out_gaze='PCR'
15:18:51 - rank 0 Shape runs [1 2 3]: [(41, 87379), (41, 86278), (41, 79071)]
15:18:51 - rank 0 Loading M

In [44]:
from scipy.stats import spearmanr
def RSA_noise_ceiling(RDMs_subjects, type_ceiling: str, corr_type: str):
    if type_ceiling not in ("upper", "lower"):
        raise ValueError(f"type_ceiling must be 'upper' or 'lower', got '{type_ceiling}'")
    if corr_type == "pearson":
        corr_fn = lambda a, b: np.corrcoef(a, b)[0, 1]
    elif corr_type == "spearman":
        corr_fn = lambda a, b: spearmanr(a, b).statistic
    else:
        raise ValueError(f"corr_type must be 'pearson' or 'spearman', got '{corr_type}'")
    
    tot_sum = sum(RDMs_subjects)
    n_sub = len(RDMs_subjects)
    rs = 0
    for RDM in RDMs_subjects:
        if type_ceiling == "upper":
            curr_avg = tot_sum / n_sub
        else:  # lower
            curr_avg = (tot_sum - RDM) / (n_sub - 1)
        rs += corr_fn(RDM, curr_avg)
    
    noise_ceiling = rs/n_sub 
    return noise_ceiling

In [45]:
noise_ceiling_estimates = []
for i in range(cfg.iterations_n):
    RDMs = []
    pseudotrials_idx = get_spaced_pseudotrials(n, cfg.pseudotrials_len_tps, cfg.pseudotrials_n, 605, model_len,)
    for n in subjects_signal:
        n_pst = n[pseudotrials_idx]
        rsa_obj = RSA(cfg.neural_metric)
        RDM_i = rsa_obj.compute_RDM(n_pst, "signal")
        RDMs.append(RDM_i)
    lower = RSA_noise_ceiling(RDMs, "lower", "pearson")
    upper = RSA_noise_ceiling(RDMs, "upper", "pearson")
    noise_ceiling_estimates.append([lower, upper])
print(np.mean(np.array(noise_ceiling_estimates), axis=0))

[0.01595491 0.3224384 ]
